# Tutorial 03 — Regression

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/tutorials/03_regression.ipynb)

**Full-featured regression analysis in a single function call.** R-style formula syntax (`y ~ x1 + x2`), automatic diagnostics (heteroscedasticity, normality, multicollinearity), multiple methods (OLS, Ridge, Lasso, Robust), and prediction intervals.

| What you'll learn | Time |
|:------------------|:-----|
| Direct input regression | 1 min |
| Formula mode | 1 min |
| Diagnostics | 2 min |
| Prediction with intervals | 1 min |
| Multiple methods compared | 2 min |

In [ ]:
!pip install -q vectrix

## 1. Direct Input

Pass `y` and `X` directly as numpy arrays. OLS by default with automatic constant term.

In [ ]:
import numpy as np
from vectrix import regress

np.random.seed(42)
X = np.random.randn(100, 2)
y = 3 + 2 * X[:, 0] - 1.5 * X[:, 1] + np.random.randn(100) * 0.5

model = regress(y=y, X=X)

## 2. Formula Mode

With a DataFrame, use R-style formulas for a more natural interface.

In [ ]:
import pandas as pd
from vectrix import regress

df = pd.DataFrame({
    "sales": [100, 150, 200, 180, 250, 300, 280, 350, 400, 380],
    "ads": [10, 15, 20, 18, 25, 30, 28, 35, 40, 38],
    "price": [50, 48, 45, 47, 42, 40, 41, 38, 35, 36],
    "promo": [0, 0, 1, 0, 1, 1, 0, 1, 1, 1],
})

model = regress(data=df, formula="sales ~ ads + price + promo")

### Formula Syntax Variations

```python
regress(data=df, formula="y ~ x1 + x2")       # Specific variables
regress(data=df, formula="y ~ .")              # All other columns
regress(data=df, formula="y ~ x1 * x2")       # With interaction term
regress(data=df, formula="y ~ x + I(x**2)")   # Polynomial terms
```

## 3. Result Object

Access all regression statistics directly.

In [ ]:
print(f"R-squared:     {model.rSquared:.4f}")
print(f"Adj R-squared: {model.adjRSquared:.4f}")
print(f"F-statistic:   {model.fStat:.2f}")
print(f"Durbin-Watson: {model.durbinWatson:.4f}")
print(f"Coefficients:  {model.coefficients}")
print(f"P-values:      {model.pvalues}")

## 4. Diagnostics

Four standard regression diagnostic tests: VIF, Breusch-Pagan, Jarque-Bera, Durbin-Watson.

In [ ]:
print(model.diagnose())

### Diagnostic Tests

| Test | What It Checks | Warning Threshold |
|------|---------------|-------------------|
| **VIF** | Multicollinearity between predictors | VIF > 10 is problematic |
| **Breusch-Pagan** | Non-constant variance (heteroscedasticity) | p-value below 0.05 |
| **Jarque-Bera** | Normality of residuals | p-value below 0.05 |
| **Durbin-Watson** | Autocorrelation in residuals | Far from 2.0 |

## 5. Prediction with Intervals

In [ ]:
new_data = pd.DataFrame({
    "ads": [50, 75, 100],
    "price": [30, 25, 20],
    "promo": [1, 1, 0],
})

predictions = model.predict(new_data)
predictions

## 6. Regression Methods

Five methods available. Switch by setting the `method` parameter.

In [ ]:
import numpy as np
import pandas as pd
from vectrix import regress

np.random.seed(42)
n = 200
big_df = pd.DataFrame({
    "marketing": np.random.randn(n) * 10 + 50,
    "price": np.random.randn(n) * 5 + 30,
    "season": np.random.choice([0, 1], n),
})
big_df["revenue"] = 100 + 8 * big_df["marketing"] - 5 * big_df["price"] + 20 * big_df["season"] + np.random.randn(n) * 10

methods = ["ols", "ridge", "lasso", "huber", "quantile"]
for m in methods:
    r = regress(data=big_df, formula="revenue ~ marketing + price + season", method=m, summary=False)
    print(f"{m:>8s}  R²={r.rSquared:.4f}  coeffs={r.coefficients.round(2)}")

### When to Use Each Method

| Method | Use Case |
|--------|----------|
| `ols` | Default, well-behaved data |
| `ridge` | Correlated predictors (L2 regularization) |
| `lasso` | Feature selection, sparse models (L1 regularization) |
| `huber` | Outliers in the data (robust loss) |
| `quantile` | Median regression, skewed distributions |

## 7. Complete Example

In [ ]:
import pandas as pd
import numpy as np
from vectrix import regress

np.random.seed(42)
n = 200
df = pd.DataFrame({
    "marketing": np.random.randn(n) * 10 + 50,
    "price": np.random.randn(n) * 5 + 30,
    "season": np.random.choice([0, 1], n),
})
df["revenue"] = 100 + 8 * df["marketing"] - 5 * df["price"] + 20 * df["season"] + np.random.randn(n) * 10

model = regress(data=df, formula="revenue ~ marketing + price + season")
print(model.diagnose())

future = pd.DataFrame({
    "marketing": [60, 70, 80],
    "price": [28, 25, 22],
    "season": [1, 0, 1],
})
print("\nPredictions:")
print(model.predict(future))

---

**Next:** [Tutorial 04 — 30+ Models](04_models.ipynb)